In [3]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
df = pd.read_parquet("df_combined.parquet")

In [5]:
print(df.shape)
df.head()

(13305915, 46)


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,year_pin_last_changed,card_on_dark_web,mcc_description,is_fraud,year,month,day,hour,day_of_week,is_weekend
0,7475327,2010-01-01 00:01:00,1556,2972,77.00,Swipe Transaction,59935,Beulah,ND,58523,...,2008,No,Miscellaneous Food Stores,0,2010,1,1,0,Friday,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,...,2015,No,Department Stores,0,2010,1,1,0,Friday,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,...,2008,No,Money Transfer,0,2010,1,1,0,Friday,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,...,2006,No,Money Transfer,0,2010,1,1,0,Friday,0
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,...,2014,No,Drinking Places (Alcoholic Beverages),0,2010,1,1,0,Friday,0


In [6]:
df = df.copy()

# varsa gereksiz index kolonu sil
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

rename_map = {}

if "id" in df.columns and "transaction_id" not in df.columns:
    rename_map["id"] = "transaction_id"

if "date" in df.columns and "transaction_datetime" not in df.columns:
    rename_map["date"] = "transaction_datetime"

if "zip" in df.columns and "merchant_zip" not in df.columns:
    rename_map["zip"] = "merchant_zip"

df = df.rename(columns=rename_map)

print(df.columns.tolist())
print(df.shape)

['transaction_id', 'transaction_datetime', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'merchant_zip', 'mcc', 'errors', 'is_return', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'client_id_card', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'mcc_description', 'is_fraud', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend']
(13305915, 46)


In [7]:
# id benzeri kolonlar string olsun
df["transaction_id"] = df["transaction_id"].astype(str)
df["client_id"] = df["client_id"].astype(str)
df["card_id"] = df["card_id"].astype(str)

# datetime
df["transaction_datetime"] = pd.to_datetime(df["transaction_datetime"], errors="coerce")

# sayısal kolonlar
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["is_fraud"] = pd.to_numeric(df["is_fraud"], errors="coerce").fillna(0).astype(int)

money_cols = ["credit_limit", "yearly_income", "per_capita_income", "total_debt"]

for col in money_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[^0-9\.\-]", "", regex=True)
            .replace("", np.nan)
            .pipe(pd.to_numeric, errors="coerce")
        )

print(df.dtypes.head(15))
print(df["is_fraud"].value_counts(dropna=False))

transaction_id                  object
transaction_datetime    datetime64[ns]
client_id                       object
card_id                         object
amount                         float64
use_chip                        object
merchant_id                      int64
merchant_city                   object
merchant_state                  object
merchant_zip                    object
mcc                             object
errors                          object
is_return                         bool
current_age                      int64
retirement_age                   int64
dtype: object
is_fraud
0    13292583
1       13332
Name: count, dtype: int64


In [8]:
#Bu adımda yaptığımız şey:

#tek bir tarih kolonunu, modelin anlayacağı birden fazla anlamlı kolona çevirmek
df["hour"] = df["transaction_datetime"].dt.hour
df["day"] = df["transaction_datetime"].dt.day
df["month"] = df["transaction_datetime"].dt.month
df["year"] = df["transaction_datetime"].dt.year
df["day_of_week_num"] = df["transaction_datetime"].dt.dayofweek

if "day_of_week" not in df.columns:
    df["day_of_week"] = df["transaction_datetime"].dt.day_name()

df["is_weekend"] = (df["day_of_week_num"] >= 5).astype(int)
df["is_night"] = ((df["hour"] < 6) | (df["hour"] > 22)).astype(int)
df["is_peak_hour"] = df["hour"].isin([12, 13, 18, 19, 20]).astype(int)

df[["transaction_datetime", "hour", "day", "month", "day_of_week", "is_weekend", "is_night"]].head()

,transaction_datetime,hour,day,month,day_of_week,is_weekend,is_night
0,2010-01-01 00:01:00,0,1,1,Friday,0,1
1,2010-01-01 00:02:00,0,1,1,Friday,0,1
2,2010-01-01 00:02:00,0,1,1,Friday,0,1
3,2010-01-01 00:05:00,0,1,1,Friday,0,1
4,2010-01-01 00:06:00,0,1,1,Friday,0,1


In [9]:
#Burada işlem tutarından türeyen feature’ları üretiyoruz.
df["log_amount"] = np.log1p(df["amount"].clip(lower=0))

amount_mean = df["amount"].mean()
amount_std = df["amount"].std()

if pd.notna(amount_std) and amount_std != 0:
    df["amount_zscore"] = (df["amount"] - amount_mean) / amount_std
else:
    df["amount_zscore"] = 0

df["high_amount"] = (df["amount"] > df["amount"].quantile(0.95)).astype(int)

if "credit_limit" in df.columns:
    df["amount_to_limit_ratio"] = np.where(
        df["credit_limit"].fillna(0) > 0,
        df["amount"] / df["credit_limit"],
        np.nan
    )
else:
    df["amount_to_limit_ratio"] = np.nan

df[["amount", "log_amount", "amount_zscore", "high_amount", "amount_to_limit_ratio"]].head()

,amount,log_amount,amount_zscore,high_amount,amount_to_limit_ratio
0,77.00,4.356709,0.316447,0,1.400000
1,14.57,2.745346,-0.511013,0,0.001601
2,80.00,4.394449,0.356210,0,0.005405
3,200.00,5.303305,1.946715,1,0.005314
4,46.41,3.858833,-0.088999,0,0.002428


In [10]:
#use_chip kolonundan:

#online mı
#chip kullanılmış mı
#swipe mı

#bilgilerini çıkarıyoruz.
if "use_chip" in df.columns:
    use_chip_str = df["use_chip"].astype(str).str.lower()
    df["is_online"] = use_chip_str.str.contains("online", na=False).astype(int)
    df["is_chip_used"] = use_chip_str.str.contains("chip", na=False).astype(int)
    df["is_swipe"] = use_chip_str.str.contains("swipe", na=False).astype(int)
else:
    df["is_online"] = 0
    df["is_chip_used"] = 0
    df["is_swipe"] = 0

df[["use_chip", "is_online", "is_chip_used", "is_swipe"]].head()

,use_chip,is_online,is_chip_used,is_swipe
0,Swipe Transaction,0,0,1
1,Swipe Transaction,0,0,1
2,Swipe Transaction,0,0,1
3,Swipe Transaction,0,0,1
4,Swipe Transaction,0,0,1


In [11]:
#Müşteri davranış feature’ları

#Burada müşterinin kendi geçmişine göre feature üretiyoruz.

#toplam işlem sayısı
#ortalama işlem tutarı
#sapma
#çok kısa sürede işlem yapmış mı
#son 3 işlem ortalaması

df = df.sort_values(["client_id", "transaction_datetime"]).reset_index(drop=True)

df["user_tx_count"] = df.groupby("client_id")["transaction_id"].transform("count")
df["user_mean_amount"] = df.groupby("client_id")["amount"].transform("mean")
df["user_std_amount"] = df.groupby("client_id")["amount"].transform("std")
df["amount_deviation"] = df["amount"] - df["user_mean_amount"]

df["time_diff"] = (
    df.groupby("client_id")["transaction_datetime"]
      .diff()
      .dt.total_seconds()
)

df["fast_tx"] = (df["time_diff"] < 60).astype(int)
df["very_fast_tx"] = (df["time_diff"] < 10).astype(int)

df["rolling_mean_3"] = (
    df.groupby("client_id")["amount"]
      .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

df["rolling_std_3"] = (
    df.groupby("client_id")["amount"]
      .transform(lambda x: x.rolling(3, min_periods=1).std())
)

df["rolling_std_3"] = df["rolling_std_3"].fillna(0)
df["rolling_amount_deviation"] = df["amount"] - df["rolling_mean_3"]

df[[
    "client_id", "amount", "user_mean_amount", "amount_deviation",
    "time_diff", "fast_tx", "very_fast_tx",
    "rolling_mean_3", "rolling_std_3", "rolling_amount_deviation"
]].head(10)

,client_id,amount,user_mean_amount,amount_deviation,time_diff,fast_tx,very_fast_tx,rolling_mean_3,rolling_std_3,rolling_amount_deviation
0,0,33.96,61.03319,-27.07319,NaN,0,0,33.960000,0.000000,0.000000
1,0,7.78,61.03319,-53.25319,23340.0,0,0,20.870000,18.512056,-13.090000
2,0,65.86,61.03319,4.82681,9240.0,0,0,35.866667,29.086907,29.993333
3,0,55.85,61.03319,-5.18319,53700.0,0,0,43.163333,31.048917,12.686667
4,0,1.37,61.03319,-59.66319,95760.0,0,0,41.026667,34.706461,-39.656667
5,0,39.91,61.03319,-21.12319,22080.0,0,0,32.376667,28.010372,7.533333
6,0,167.62,61.03319,106.58681,39420.0,0,0,69.633333,87.019383,97.986667
7,0,28.68,61.03319,-32.35319,15240.0,0,0,78.736667,77.179748,-50.056667
8,0,30.32,61.03319,-30.71319,960.0,0,0,75.540000,79.747835,-45.220000
9,0,2.34,61.03319,-58.69319,4800.0,0,0,20.446667,15.702259,-18.106667


In [12]:
df["client_id"].nunique()

1219

In [13]:
#Merchant risk feature

#Bazı MCC kategorilerinde fraud oranı daha yüksek olabilir.
#Bunu tek bir merchant_risk_flag değişkenine çeviriyoruz.

if "mcc" in df.columns:
    mcc_stats = (
        df.groupby("mcc")["is_fraud"]
          .agg(["mean", "count"])
          .reset_index()
          .rename(columns={"mean": "mcc_fraud_rate", "count": "mcc_total_count"})
    )

    global_fraud_rate = df["is_fraud"].mean()

    mcc_stats["merchant_risk_flag"] = np.where(
        (mcc_stats["mcc_total_count"] >= 100) &
        (mcc_stats["mcc_fraud_rate"] > global_fraud_rate * 2),
        1,
        0
    )

    df = df.merge(
        mcc_stats[["mcc", "merchant_risk_flag"]],
        on="mcc",
        how="left"
    )

    df["merchant_risk_flag"] = df["merchant_risk_flag"].fillna(0).astype(int)
else:
    df["merchant_risk_flag"] = 0

df[["mcc", "merchant_risk_flag"]].head()

,mcc,merchant_risk_flag
0,5942,0
1,5812,0
2,5813,0
3,5211,0
4,5499,0


In [14]:
mcc_stats.sort_values("mcc_fraud_rate", ascending=False).head(10)

,mcc,mcc_fraud_rate,mcc_total_count,merchant_risk_flag
38,4411,0.385514,428,1
68,5733,0.238245,319,1
4,3006,0.082621,351,1
46,5045,0.073040,2793,1
12,3144,0.068862,334,1
67,5732,0.057453,6997,1
3,3005,0.056266,391,1
7,3009,0.053922,408,1
47,5094,0.046718,5180,1
5,3007,0.044619,381,1


In [15]:
#Numerik boş değerleri temizleme
num_cols_all = df.select_dtypes(include=[np.number]).columns
df[num_cols_all] = df[num_cols_all].fillna(0)

print(df.isna().sum().sort_values(ascending=False).head(20))

errors                   13094522
merchant_state            1563700
transaction_id                  0
day_of_week                     0
log_amount                      0
is_peak_hour                    0
is_night                        0
day_of_week_num                 0
is_weekend                      0
hour                            0
high_amount                     0
day                             0
month                           0
year                            0
is_fraud                        0
mcc_description                 0
card_on_dark_web                0
amount_zscore                   0
amount_to_limit_ratio           0
acct_open_date                  0
dtype: int64


In [16]:
df["errors"] = df["errors"].fillna("no_error")

In [17]:
df["merchant_state"] = df["merchant_state"].fillna("Unknown")

In [18]:
#MVP model tablosunu oluşturma
feature_cols = [
    "amount",
    "log_amount",
    "amount_zscore",
    "high_amount",
    "amount_to_limit_ratio",
    "hour",
    "day",
    "month",
    "year",
    "day_of_week",
    "day_of_week_num",
    "is_weekend",
    "is_night",
    "is_peak_hour",
    "use_chip",
    "is_online",
    "is_chip_used",
    "is_swipe",
    "merchant_city",
    "merchant_state",
    "mcc",
    "merchant_risk_flag",
    "is_return",
    "current_age",
    "gender",
    "yearly_income",
    "per_capita_income",
    "total_debt",
    "credit_score",
    "user_tx_count",
    "user_mean_amount",
    "user_std_amount",
    "amount_deviation",
    "time_diff",
    "fast_tx",
    "very_fast_tx",
    "rolling_mean_3",
    "rolling_std_3",
    "rolling_amount_deviation"
]

feature_cols = [c for c in feature_cols if c in df.columns]

df_model = df[feature_cols + ["is_fraud"]].copy()

print("Model tablosu shape:", df_model.shape)
print(df_model["is_fraud"].value_counts(dropna=False))

Model tablosu shape: (13305915, 40)
is_fraud
0    13292583
1       13332
Name: count, dtype: int64


In [19]:
#MVP samlpe üret , yani tüm fraud kayıtlarını koru.
fraud_df = df_model[df_model["is_fraud"] == 1].copy()
nonfraud_df = df_model[df_model["is_fraud"] == 0].copy()

print("Fraud satır sayısı:", len(fraud_df))
print("Non-fraud satır sayısı:", len(nonfraud_df))

# Yaklaşık 1:20 oranı -> fraud oranı yaklaşık %5
desired_nonfraud = min(len(nonfraud_df), len(fraud_df) * 20)

nonfraud_sample = nonfraud_df.sample(n=desired_nonfraud, random_state=42)

df_mvp = pd.concat([fraud_df, nonfraud_sample], axis=0)
df_mvp = df_mvp.sample(frac=1, random_state=42).reset_index(drop=True)

print("MVP sample shape:", df_mvp.shape)
print(df_mvp["is_fraud"].value_counts())
print(df_mvp["is_fraud"].value_counts(normalize=True))

Fraud satır sayısı: 13332
Non-fraud satır sayısı: 13292583
MVP sample shape: (279972, 40)
is_fraud
0    266640
1     13332
Name: count, dtype: int64
is_fraud
0    0.952381
1    0.047619
Name: proportion, dtype: float64


In [20]:
#Train / Test split
X = df_mvp.drop(columns=["is_fraud"])
y = df_mvp["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train fraud oranı:", y_train.mean())
print("Test fraud oranı:", y_test.mean())

Train shape: (223977, 39)
Test shape: (55995, 39)
Train fraud oranı: 0.047620961080825266
Test fraud oranı: 0.047611393874453074


In [21]:
# Sadece TRAIN ' de RESAMPLING 

In [22]:
train_df = X_train.copy()
train_df["is_fraud"] = y_train.values

train_fraud = train_df[train_df["is_fraud"] == 1].copy()
train_nonfraud = train_df[train_df["is_fraud"] == 0].copy()

desired_train_nonfraud = min(len(train_nonfraud), len(train_fraud) * 20)

train_nonfraud_sample = train_nonfraud.sample(
    n=desired_train_nonfraud,
    random_state=42
)

train_balanced = pd.concat([train_fraud, train_nonfraud_sample], axis=0)
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

X_train_balanced = train_balanced.drop(columns=["is_fraud"])
y_train_balanced = train_balanced["is_fraud"]

print("Balanced train shape:", X_train_balanced.shape)
print(y_train_balanced.value_counts())
print(y_train_balanced.value_counts(normalize=True))

Balanced train shape: (223977, 39)
is_fraud
0    213311
1     10666
Name: count, dtype: int64
is_fraud
0    0.952379
1    0.047621
Name: proportion, dtype: float64


In [23]:
# Preprocessing pipeline hazırlamak
num_cols = X_train_balanced.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train_balanced.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerik kolon sayısı:", len(num_cols))
print("Kategorik kolon sayısı:", len(cat_cols))
print("Kategorik kolonlar:", cat_cols)

Numerik kolon sayısı: 32
Kategorik kolon sayısı: 7
Kategorik kolonlar: ['day_of_week', 'use_chip', 'merchant_city', 'merchant_state', 'mcc', 'is_return', 'gender']


In [24]:
#sayısal ve kategorik veriyi modele uygun formata hazırlamak
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [25]:
#MVP modelini eğitelim
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train_balanced, y_train_balanced)
print("Model eğitildi.")

Model eğitildi.


In [26]:
#Modeli Değerlendirme
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nPrecision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Average Precision:", average_precision_score(y_test, y_proba))

#buradaki amaç modelimiz gerçekten fraud yakalıyor mu ? 


Confusion Matrix:
[[51571  1758]
 [  158  2508]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     53329
           1       0.59      0.94      0.72      2666

    accuracy                           0.97     55995
   macro avg       0.79      0.95      0.85     55995
weighted avg       0.98      0.97      0.97     55995


Precision: 0.5879043600562588
Recall: 0.940735183795949
F1: 0.723600692440854
ROC AUC: 0.9889760735482865
Average Precision: 0.8978791612358744


In [27]:
joblib.dump(model, "../fraud_mvp_model.joblib")
print("Model kaydedildi: ../fraud_mvp_model.joblib")

Model kaydedildi: ../fraud_mvp_model.joblib


In [28]:
# Takım arkadaşlarının modeli yükleyip kullanması

loaded_model = joblib.load("../fraud_mvp_model.joblib")

sample_preds = loaded_model.predict(X_test.head(10))
sample_proba = loaded_model.predict_proba(X_test.head(10))[:, 1]

result_df = X_test.head(10).copy()
result_df["predicted_fraud"] = sample_preds
result_df["fraud_probability"] = sample_proba

result_df

,amount,log_amount,amount_zscore,high_amount,amount_to_limit_ratio,hour,day,month,year,day_of_week,...,user_std_amount,amount_deviation,time_diff,fast_tx,very_fast_tx,rolling_mean_3,rolling_std_3,rolling_amount_deviation,predicted_fraud,fraud_probability
213278,58.00,4.077537,0.064617,0,1.487179,16,4,6,2019,Tuesday,...,13.770190,38.092672,36960.0,0,0,36.350000,20.748135,21.650000,1,0.993419
11189,2.39,1.220830,-0.672449,0,0.000138,9,8,11,2015,Sunday,...,69.851204,-77.342146,1800.0,0,0,37.960000,53.920531,-35.570000,0,0.001211
69347,40.63,3.728821,-0.165608,0,0.005079,0,21,5,2018,Monday,...,63.806028,-19.781255,300060.0,0,0,56.920000,27.783220,-16.290000,0,0.000014
106310,35.38,3.594019,-0.235193,0,0.001264,16,29,3,2016,Tuesday,...,86.734909,-51.016861,23640.0,0,0,121.673333,119.251371,-86.293333,0,0.000750
135293,76.00,4.343805,0.303193,0,0.013103,11,27,1,2012,Friday,...,49.637590,30.307858,1080.0,0,0,65.313333,18.509850,10.686667,0,0.000294
200380,53.25,3.993603,0.001660,0,0.003647,19,22,11,2016,Tuesday,...,68.559638,8.776747,8520.0,0,0,34.683333,25.158763,18.566667,0,0.045963
106752,61.60,4.136765,0.112333,0,0.004726,3,6,11,2011,Sunday,...,42.466616,27.517548,13200.0,0,0,70.940000,34.278022,-9.340000,0,0.033354
50640,54.51,4.016563,0.018360,0,0.005516,15,16,3,2019,Saturday,...,35.533832,-5.593513,11580.0,0,0,64.046667,16.414178,-9.536667,0,0.012213
86384,3.01,1.388791,-0.664232,0,0.000310,12,15,1,2013,Tuesday,...,61.865381,-28.274548,15420.0,0,0,7.013333,4.103417,-4.003333,0,0.007261
27930,19.24,3.007661,-0.449116,0,0.001115,6,31,8,2011,Wednesday,...,127.674328,-49.139395,58440.0,0,0,55.450000,67.315394,-36.210000,0,0.006252


In [29]:
df_mvp.to_parquet("../data/processed/transaction_features_mvp_sample.parquet", index=False)

In [30]:
import sys
import sklearn

print(sys.executable)
print(sklearn.__version__)

/Users/emre/.pyenv/versions/3.12.9/bin/python
1.6.1


In [31]:
import joblib
import sklearn
import sys

print(sys.executable)
print(sklearn.__version__)

joblib.dump(model, "../models/fraud_mvp_model.joblib")
print("Model yeniden kaydedildi: ../models/fraud_mvp_model.joblib")

/Users/emre/.pyenv/versions/3.12.9/bin/python
1.6.1
Model yeniden kaydedildi: ../models/fraud_mvp_model.joblib


In [32]:
df_mvp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279972 entries, 0 to 279971
Data columns (total 40 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   amount                    279972 non-null  float64
 1   log_amount                279972 non-null  float64
 2   amount_zscore             279972 non-null  float64
 3   high_amount               279972 non-null  int64  
 4   amount_to_limit_ratio     279972 non-null  float64
 5   hour                      279972 non-null  int32  
 6   day                       279972 non-null  int32  
 7   month                     279972 non-null  int32  
 8   year                      279972 non-null  int32  
 9   day_of_week               279972 non-null  object 
 10  day_of_week_num           279972 non-null  int32  
 11  is_weekend                279972 non-null  int64  
 12  is_night                  279972 non-null  int64  
 13  is_peak_hour              279972 non-null  i